# 01 — Télécharger, nettoyer et découper French Alpaca

Ce notebook prépare trois partitions **disjointes** du dataset `jpacifico/French-Alpaca-dataset-Instruct-55K` :

- 40 % pour `fr_bau` : instruction/input en français, réponse à traduire en baoulé ;
- 40 % pour `bau_bau` : instruction/input et réponse à traduire en baoulé ;
- 20 % pour `fr_fr` : exemple conservé en français.

Le nettoyage reste volontairement conservateur : normalisation Unicode NFC, normalisation des fins de ligne, suppression des lignes sans instruction ou sans réponse, puis déduplication exacte. Aucun texte n'est reformulé.

In [ ]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import hashlib
import json
import random
import unicodedata
import urllib.request

DATASET_ID = "jpacifico/French-Alpaca-dataset-Instruct-55K"
SOURCE_FILENAME = "Acquiesce_data_55k_instructions.json"
SOURCE_URL = (
    "https://huggingface.co/datasets/"
    f"{DATASET_ID}/resolve/main/{SOURCE_FILENAME}?download=true"
)
SEED = 20260922

PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "00_raw"
SPLIT_DIR = PROJECT_ROOT / "data" / "01_splits"
RAW_PATH = RAW_DIR / SOURCE_FILENAME

RAW_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

print("Répertoire du projet :", PROJECT_ROOT)
print("Graine du découpage   :", SEED)

## 1. Télécharger le fichier source

Le téléchargement est ignoré si le fichier existe déjà, ce qui permet de reprendre le notebook sans télécharger deux fois les 26 Mo.

In [ ]:
if RAW_PATH.exists():
    print(f"Fichier déjà présent : {RAW_PATH} ({RAW_PATH.stat().st_size:,} octets)")
else:
    request = urllib.request.Request(
        SOURCE_URL,
        headers={"User-Agent": "Lree-AI-Labs-dataset-preparation/1.0"},
    )
    temporary_path = RAW_PATH.with_suffix(RAW_PATH.suffix + ".part")
    print("Téléchargement depuis :", SOURCE_URL)
    with urllib.request.urlopen(request, timeout=120) as response, temporary_path.open("wb") as target:
        while chunk := response.read(1024 * 1024):
            target.write(chunk)
    temporary_path.replace(RAW_PATH)
    print(f"Téléchargement terminé : {RAW_PATH.stat().st_size:,} octets")

source_sha256 = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()
print("SHA-256 :", source_sha256)

## 2. Charger et auditer la structure

On vérifie que chaque ligne possède les champs `instruction`, `input` et `output`. Le champ `input` a le droit d'être vide.

In [ ]:
with RAW_PATH.open(encoding="utf-8") as handle:
    raw_records = json.load(handle)

if not isinstance(raw_records, list):
    raise TypeError(f"Le JSON doit contenir une liste, pas {type(raw_records).__name__}.")

required_fields = {"instruction", "input", "output"}
missing_field_counts = Counter()
for record in raw_records:
    if isinstance(record, dict):
        missing_field_counts.update(required_fields - set(record))
    else:
        missing_field_counts["<ligne_non_objet>"] += 1

print(f"Lignes chargées : {len(raw_records):,}")
print("Champs manquants :", dict(missing_field_counts) or "aucun")
print("Premier exemple :")
print(json.dumps(raw_records[0], ensure_ascii=False, indent=2))

## 3. Normaliser, filtrer et dédupliquer

La normalisation NFC est préférable à une normalisation agressive : elle harmonise les représentations Unicode sans supprimer les accents ou futurs signes diacritiques utiles au baoulé.

In [ ]:
def normalize_text(value):
    if value is None:
        return ""
    if not isinstance(value, str):
        value = str(value)
    value = value.replace("\r\n", "\n").replace("\r", "\n")
    return unicodedata.normalize("NFC", value).strip()

def content_key(record):
    payload = [record["instruction_fr"], record["input_fr"], record["output_fr"]]
    return hashlib.sha256(
        json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    ).hexdigest()

clean_records = []
seen = set()
audit = Counter()

for source_row_index, raw in enumerate(raw_records):
    if not isinstance(raw, dict):
        audit["non_object"] += 1
        continue

    record = {
        "instruction_fr": normalize_text(raw.get("instruction")),
        "input_fr": normalize_text(raw.get("input")),
        "output_fr": normalize_text(raw.get("output")),
    }

    if not record["instruction_fr"]:
        audit["empty_instruction"] += 1
        continue
    if not record["output_fr"]:
        audit["empty_output"] += 1
        continue

    key = content_key(record)
    if key in seen:
        audit["exact_duplicate"] += 1
        continue
    seen.add(key)

    clean_records.append({
        "id": f"french_alpaca_{key[:16]}",
        "source_row_index": source_row_index,
        "source_dataset": DATASET_ID,
        **record,
    })

print(f"Lignes source       : {len(raw_records):,}")
print(f"Lignes conservées   : {len(clean_records):,}")
print("Lignes écartées     :", dict(audit) or "aucune")
assert len({record['id'] for record in clean_records}) == len(clean_records)

## 4. Effectuer le découpage déterministe 40/40/20

La graine fixe rend le découpage reproductible. Les deux groupes à traduire reçoivent chacun `round(N × 0,40)` lignes ; le reste est attribué à `fr_fr`.

In [ ]:
shuffled = clean_records.copy()
random.Random(SEED).shuffle(shuffled)

n_total = len(shuffled)
n_fr_bau = round(n_total * 0.40)
n_bau_bau = round(n_total * 0.40)
n_fr_fr = n_total - n_fr_bau - n_bau_bau

partitions = {
    "fr_bau": shuffled[:n_fr_bau],
    "bau_bau": shuffled[n_fr_bau:n_fr_bau + n_bau_bau],
    "fr_fr": shuffled[n_fr_bau + n_bau_bau:],
}

for direction, records in partitions.items():
    for record in records:
        record["direction"] = direction

for direction, records in partitions.items():
    print(f"{direction:8s}: {len(records):6,d} ({len(records) / n_total:.3%})")

## 5. Exporter les partitions et leur manifeste

À ce stade, tous les textes sont encore en français. Les suffixes `_fr` évitent de confondre une source non traduite avec une véritable donnée baoulé. Le prochain notebook produira les champs traduits.

In [ ]:
def write_jsonl(path, records):
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    with temporary_path.open("w", encoding="utf-8", newline="\n") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
    temporary_path.replace(path)

output_paths = {}
for direction, records in partitions.items():
    filename = f"{direction}_source.jsonl"
    output_path = SPLIT_DIR / filename
    write_jsonl(output_path, records)
    output_paths[direction] = str(output_path.relative_to(PROJECT_ROOT))
    print(f"Écrit : {output_path} ({len(records):,} lignes)")

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_dataset": DATASET_ID,
    "source_url": SOURCE_URL,
    "source_filename": SOURCE_FILENAME,
    "source_sha256": source_sha256,
    "license": "Apache-2.0",
    "attribution": "Dataset créé par Jonathan Pacifico",
    "seed": SEED,
    "source_row_count": len(raw_records),
    "clean_row_count": n_total,
    "discarded": dict(audit),
    "partitions": {
        direction: {
            "rows": len(records),
            "fraction": len(records) / n_total,
            "path": output_paths[direction],
        }
        for direction, records in partitions.items()
    },
}
manifest_path = SPLIT_DIR / "split_manifest.json"
manifest_path.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print("Manifeste :", manifest_path)

## 6. Vérifications finales

Ces assertions empêchent notamment qu'un exemple apparaisse dans plusieurs directions.

In [ ]:
id_sets = {
    direction: {record["id"] for record in records}
    for direction, records in partitions.items()
}

assert not (id_sets["fr_bau"] & id_sets["bau_bau"])
assert not (id_sets["fr_bau"] & id_sets["fr_fr"])
assert not (id_sets["bau_bau"] & id_sets["fr_fr"])
assert sum(map(len, id_sets.values())) == n_total
assert set().union(*id_sets.values()) == {record["id"] for record in clean_records}

for direction, path in output_paths.items():
    with (PROJECT_ROOT / path).open(encoding="utf-8") as handle:
        exported_count = sum(1 for line in handle if line.strip())
    assert exported_count == len(partitions[direction])

print("✓ Partitions disjointes")
print("✓ Toutes les lignes nettoyées ont été attribuées")
print("✓ Comptages des fichiers exportés validés")

## Résultat de cette étape

Le répertoire `data/01_splits/` contient :

- `fr_bau_source.jsonl` ;
- `bau_bau_source.jsonl` ;
- `fr_fr_source.jsonl` ;
- `split_manifest.json`.

Le prochain notebook traitera la traduction avec cache, reprise sur erreur et contrôle qualité.